In [ ]:
print("Ok")

In [ ]:
import pandas as pd 

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Pinecone
import pinecone
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.llms import CTransformers


## Extract data from the PDF 

In [ ]:
import os 
print(os.getcwd())

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
def load_data(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [ ]:
extracted_data = load_data("../data/")

In [ ]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks



In [ ]:
text_chunks = text_split(extracted_data)
print("Length of chunk:", len(text_chunks))

In [ ]:
text_chunks

### Download the embedding model 

In [ ]:
def download_hugging_face_embeddings():
    embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embedding

In [ ]:
embeddings = download_hugging_face_embeddings()
# it take time ...!

In [ ]:
embeddings

In [ ]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

In [ ]:
query_result

In [ ]:
import os 
from dotenv import load_dotenv
from pinecone import Pinecone 


load_dotenv()
pc = Pinecone(api_key="PINECONE_API_KEY")

In [ ]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv(override=True) # Use override to ensure fresh values
api_key = os.getenv("PINECONE_API_KEY").strip() # .strip() removes accidental spaces

pc = Pinecone(api_key=api_key)

try:
    print("Connecting...")
    indices = pc.list_indexes().names()
    print(f"Success! Found: {indices}")
except Exception as e:
    print(f"Connection failed: {e}")


In [ ]:
from pinecone import ServerlessSpec

pc.create_index(
    name="medical-chatbot",
    dimension=384, # Dimension for all-MiniLM-L6-v2
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1") # Adjust region if needed
)


In [ ]:
# Fetch and print only the names of your indexes
index_names = pc.list_indexes().names()
print(index_names)

# Check specifically for your medical chatbot
if "medical-chatbot" in index_names:
    print("Found it!")


In [ ]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name="medical-chatbot",
    embedding=embeddings
)


In [ ]:
# Connect to the index specifically for data operations
index = pc.Index("medical-chatbot")
stats = index.describe_index_stats()

print(f"Total Vectors in Index: {stats['total_vector_count']}")


In [ ]:
## oops the index is empty , so let add some data

docsearch = PineconeVectorStore.from_texts(
    texts=[t.page_content for t in text_chunks],
    embedding=embeddings,
    index_name="medical-chatbot"
)

In [ ]:

query = "What are Allergies"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

In [ ]:
prompt_template = """
Use the context below to answer the question.

If the context contains relevant information, explain it clearly and completely.
If the context is partially relevant, still try to answer using it.
Only say "I don't know" if there is absolutely no relevant information in the context.

Context:
{context}

Question:
{question}

Provide a clear and helpful answer:

"""

In [ ]:
docs = docsearch.as_retriever(search_kwargs={'k': 2}).invoke("what is hypoxemia?")
print(docs)


In [ ]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="llama3",
    temperature=0.8
)

In [ ]:
from langchain_classic.chains import RetrievalQA


qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={'k': 2}),
    return_source_documents = True,
    # chain_type_kwargs=chain_type_kwargs
)

In [ ]:
while True:
    user_input = input(f'Input Prompt:')
    result=qa({"query":user_input})
    print("Response: ", result['result'] )
